In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import datasets

knowledge_base = datasets.load_dataset("m-ric/huggingface_doc", split="train")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

C:\Users\Philippe\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Philippe\.cache\huggingface\hub\datasets--m-ric--huggingface_doc. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

In [ ]:
knowledge_base

In [3]:
from tqdm import tqdm
from transformers import AutoTokenizer
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy

source_docs = [
    Document(page_content=doc["text"],
             metadata={"source": doc["source"].split("/")[1]})
    for doc in knowledge_base
]

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained("thenlper/gte-small"),
    chunk_size=200,
    chunk_overlap=20,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)

# Split docs and keep only unique ones
print("Splitting documents...")
docs_processed = []
unique_texts = {}
for doc in tqdm(source_docs):
    new_docs = text_splitter.split_documents([doc])
    for new_doc in new_docs:
        if new_doc.page_content not in unique_texts:
            unique_texts[new_doc.page_content] = True
            docs_processed.append(new_doc)

print(
    "Embedding documents... This should take a few minutes (5 minutes on MacBook with M1 Pro)"
)
embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")
vectordb = FAISS.from_documents(
    documents=docs_processed,
    embedding=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

C:\Users\Philippe\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Philippe\.cache\huggingface\hub\models--thenlper--gte-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
from smolagents import Tool
from langchain_core.vectorstores import VectorStore


class RetrieverTool(Tool):
    name = "retriever"
    description = "Using semantic similarity, retrieves some documents from the knowledge base that have the closest embeddings to the input query."
    inputs = {
        "query": {
            "type":
            "string",
            "description":
            "The query to perform. This should be semantically close to your target documents. Use the affirmative form rather than a question.",
        }
    }
    output_type = "string"

    def __init__(self, vectordb: VectorStore, **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.vectordb.similarity_search(
            query,
            k=7,
        )

        return "\nRetrieved documents:\n" + "".join([
            f"===== Document {str(i)} =====\n" + doc.page_content
            for i, doc in enumerate(docs)
        ])

In [101]:
from smolagents import OpenAIServerModel, ToolCallingAgent
from config.config import Gemini_API_KEY
from smolagents.monitoring import LogLevel

answer_llm = OpenAIServerModel(
    model_id="gemini-1.5-flash",
    # model_id="gemini-2.0-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=Gemini_API_KEY,
    temperature=0.2)

retriever_tool = RetrieverTool(vectordb)
agent = ToolCallingAgent(tools=[retriever_tool],
                         model=answer_llm,
                         verbosity_level=LogLevel.ERROR)

In [78]:
agent_output = agent.run("How can I push a model to the Hub?")

print("Final output:")
print(agent_output)

Final output:
To push a model to the Hugging Face Hub, you typically use the `push_to_hub` method.  This is often integrated into training frameworks like the Hugging Face `Trainer` class.  Before pushing, ensure you have git-lfs installed and are logged into your Hugging Face account using `huggingface-cli login`.  The `push_to_hub` method usually takes additional arguments to specify details about your model, such as the model name, tags, and a model card.


In [ ]:
eval_dataset = datasets.load_dataset("m-ric/huggingface_doc_qa_eval",
                                     split="train")

In [106]:
##### using agentic RAG to answer questions
import time

outputs_agentic_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]

    enhanced_question = f"""Using the information contained in your knowledge base, which you can access with the 'retriever' tool,
give a comprehensive answer to the question below.
Respond only to the question asked, response should be concise and relevant to the question.
If you cannot find information, do not give up and try calling your retriever again with different arguments!
Make sure to have covered the question completely by calling the retriever tool several times with semantically different queries.
Your queries should not be questions but affirmative form sentences: e.g. rather than "How do I load a model from the Hub in bf16?", query should be "load a model from the Hub bf16 weights".

Question:
{question}"""
    answer = agent.run(enhanced_question)
    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_agentic_rag.append(results_agentic)
    time.sleep(10)

  0%|          | 0/65 [00:00<?, ?it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: x86_64-unknown-linux-musl
True answer: x86_64-unknown-linux-musl


  2%|▏         | 1/65 [00:11<12:06, 11.36s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: I am sorry, but I cannot find information about the BLIP-Diffusion model in my knowledge base.  Please try a different query.
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  3%|▎         | 2/65 [00:22<11:53, 11.33s/it]

...

 97%|█████████▋| 63/65 [12:14<00:23, 11.83s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: The `equal_height` parameter, passed to the `.style()` method of `gr.Row()`, ensures that elements in a Gradio row have the same height.
True answer: equal_height


 98%|█████████▊| 64/65 [12:25<00:11, 11.77s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: pip install --upgrade-strategy eager optimum["openvino"]
True answer: pip install --upgrade-strategy eager optimum["openvino"]


100%|██████████| 65/65 [12:37<00:00, 11.65s/it]


In [107]:
##### using only RAG to answer questions

# reader_llm = OpenAIServerModel(
#     model_id="gemini-2.0-flash",
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=Gemini_API_KEY,
#     temperature=0.2)
outputs_standard_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]
    context = retriever_tool(question)

    prompt = f"""Given the question and supporting documents below, give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
Provide the number of the source document when relevant.
If you cannot find information, do not give up and try calling your retriever again with different arguments!

Question:
{question}

{context}
"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard_rag.append(results_agentic)
    time.sleep(10)

  0%|          | 0/65 [00:00<?, ?it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for the **x86_64-unknown-linux-musl** architecture (Document 0).

True answer: x86_64-unknown-linux-musl


  2%|▏         | 1/65 [00:10<11:28, 10.75s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: BLIP-Diffusion enables zero-shot subject-driven generation and control-guided zero-shot generation (Document 5).  It can also be combined with techniques like ControlNet and prompt-to-prompt for novel subject-driven generation and editing applications (Document 2).

True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  3%|▎         | 2/65 [00:21<11:19, 10.78s/it]

...

 97%|█████████▋| 63/65 [11:20<00:21, 10.71s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: The `equal_height` parameter, passed to the `.style()` method of `gr.Row()`, ensures elements in a row have the same height. (Document 0)

True answer: equal_height


 98%|█████████▊| 64/65 [11:30<00:10, 10.69s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: `pip install --upgrade-strategy eager optimum["openvino"]` (Document 1, Document 2, Document 5)

True answer: pip install --upgrade-strategy eager optimum["openvino"]


100%|██████████| 65/65 [11:41<00:00, 10.79s/it]


In [ ]:
##### using vanilla LLM to answer questions

# answer_llm = OpenAIServerModel(
#     model_id="gemini-2.0-flash",
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=Gemini_API_KEY,
#     temperature=0.2)
outputs_standard = []

for example in tqdm(eval_dataset):
    question = example["question"]

    prompt = f"""Answer the following question as clearly and concisely as possible.
Use your own knowledge to respond.
If the question is ambiguous or cannot be answered definitively, state so clearly.

Question:
{question}

"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_llm = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard.append(results_llm)
    time.sleep(5)

  0%|          | 0/65 [00:00<?, ?it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: Linux x64 using the musl libc.

True answer: x86_64-unknown-linux-musl


  2%|▏         | 1/65 [00:05<05:48,  5.45s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The purpose of the BLIP-Diffusion model is to generate images from text prompts.  It combines the strengths of a large language model (for understanding text) and a diffusion model (for image generation).

True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  3%|▎         | 2/65 [00:11<05:52,  5.59s/it]

...

Question: What is the default context window size for Local Attention in the LongT5 model?

Answer: The question is ambiguous.  LongT5 doesn't have a single, publicly documented "default" local attention context window size.  The size is likely a hyperparameter that can be varied during training or inference, and its value depends on the specific LongT5 variant and its configuration.

True answer: 127 tokens


  8%|▊         | 5/65 [00:28<05:43,  5.73s/it]

Question: What method is used to load a checkpoint for a task using `AutoPipeline`?

Answer: `AutoPipeline` doesn't directly support loading checkpoints in the same way a typical training loop might.  It's designed for automated pipeline creation and execution, not for resuming interrupted training from a specific checkpoint.  Therefore, there's no specific method.

True answer: from_pretrained()


  9%|▉         | 6/65 [00:34<05:37,  5.72s/it]

Question: What is the purpose of Diffusers library?

Answer: The Diffusers library provides tools for using and training diffusion models, primarily for image generation and manipulation.

True answer: To serve as a modular toolbox for both inference and training of state-of-the-art pretrained diffusion models across multiple modalities.


In [56]:
EVALUATION_PROMPT = """You are a fair evaluator language model.

You will be given an instruction, a response to evaluate, a reference answer that gets a score of 3, and a score rubric representing a evaluation criteria are given.
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 3. You should refer to the score rubric.
3. The output format should look as follows: \"Feedback: {{write a feedback for criteria}} [RESULT] {{an integer number between 1 and 3}}\"
4. Please do not generate any other opening, closing, and explanations. Be sure to include [RESULT] in your output.
5. Do not score conciseness: a correct answer that covers the question should receive max score, even if it contains additional useless information.

The instruction to evaluate:
{instruction}

Response to evaluate:
{response}

Reference Answer (Score 3):
{reference_answer}

Score Rubrics:
[Is the response complete, accurate, and factual based on the reference answer?]
Score 1: The response is completely incomplete, inaccurate, and/or not factual.
Score 2: The response is somewhat complete, accurate, and/or factual.
Score 3: The response is completely complete, accurate, and/or factual.

Feedback:"""

In [71]:
import pandas as pd

evaluation_llm = OpenAIServerModel(
    # model_id="gemini-2.5-flash-preview-04-17",
    model_id="gemini-2.0-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=Gemini_API_KEY,
    temperature=0)

results = {}
for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:
    print("evaluating {}".format(system_type))

    for experiment in tqdm(outputs):
        eval_prompt = EVALUATION_PROMPT.format(
            instruction=experiment["question"],
            response=experiment["generated_answer"],
            reference_answer=experiment["true_answer"],
        )
        messages = [
            {
                "role": "system",
                "content": "You are a fair evaluator language model."
            },
            {
                "role": "user",
                "content": eval_prompt
            },
        ]

        eval_result = evaluation_llm.generate(messages).content

        try:
            feedback, score = [
                item.strip() for item in eval_result.split("[RESULT]")
            ]
            experiment["eval_score_LLM_judge"] = score
            experiment["eval_feedback_LLM_judge"] = feedback
        except:
            print(f"Parsing failed - output was: {eval_result}")

        time.sleep(5)

    results[system_type] = pd.DataFrame.from_dict(outputs)
    results[system_type] = results[system_type].loc[
        ~results[system_type]["generated_answer"].str.contains("Error")]

evaluating agentic_rag


100%|██████████| 65/65 [06:06<00:00,  5.64s/it]


evaluating standard_rag


100%|██████████| 65/65 [06:06<00:00,  5.64s/it]


evaluating standard


100%|██████████| 65/65 [06:10<00:00,  5.70s/it]


In [ ]:
DEFAULT_SCORE = 2  # Give average score whenever scoring fails


def fill_score(x):
    try:
        return int(x)
    except:
        return DEFAULT_SCORE


for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:

    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge"].fillna(
            DEFAULT_SCORE).apply(fill_score))
    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge_int"] - 1) / 2

    print(
        f"Average score for {system_type} : {results[system_type]['eval_score_LLM_judge_int'].mean()*100:.1f}%"
    )

Average score for agentic_rag RAG: 84.6%
Average score for standard_rag RAG: 80.0%
Average score for standard RAG: 62.5%


In [99]:
import json

# Create a dictionary of JSON strings
json_results = {}

for system_type, df in results.items():
    json_results[system_type] = json.loads(df.to_json(orient='records'))

# Save to a file
with open('.//results//gemini-2.0-flash.json', 'w') as f:
    json.dump(json_results, f, indent=4)